In [3]:
!pip install -q transformers[torch] datasets evaluate rouge_score accelerate

import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project running on: {device}")

Project running on: cuda


In [21]:
from datasets import load_dataset

# 1. Load 10% of the data (~28,000 examples)
# Using a seed ensures that every time you run this, you get the SAME articles (Reproducibility)
raw_data = load_dataset("cnn_dailymail", "3.0.0", split="train[:10%]").shuffle(seed=42)

# 2. Split into Train (90%) and Validation (10%)
# This is the "Industry Gold Standard" for evaluating AI fairly
split_data = raw_data.train_test_split(test_size=0.1)

train_set = split_data["train"]
val_set = split_data["test"]

print(f"Final Model Strategy: Training on {len(train_set)} articles, Validating on {len(val_set)} articles.")

Final Model Strategy: Training on 25839 articles, Validating on 2872 articles.


In [22]:
model_checkpoint = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_checkpoint)

# Test it:
test_text = "AI is changing the world."
print(f"Tokens: {tokenizer.tokenize(test_text)}")
print(f"IDs: {tokenizer.encode(test_text)}")

Tokens: ['▁AI', '▁is', '▁changing', '▁the', '▁world', '.']
IDs: [7833, 19, 2839, 8, 296, 5, 1]


In [23]:
def preprocess_function(examples):
    # 1. Add the prefix "summarize: " (T5's instruction manual)
    inputs = ["summarize: " + doc for doc in examples["article"]]
    
    # 2. Tokenize inputs (The News Articles)
    model_inputs = tokenizer(
        inputs, 
        max_length=512,      # Professional standard to balance memory and context
        truncation=True,     # Cut text if it's longer than 512
        padding="max_length" # Make all articles the same size (essential for GPU math)
    )

    # 3. Tokenize targets (The Headlines)
    # We use 'text_target' because the headline is what the model is learning to write
    labels = tokenizer(
        text_target=examples["highlights"], 
        max_length=64, 
        truncation=True, 
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Now, we apply this factory to our 1% dataset
# Use the new splits we created:

tokenized_train = train_set.map(preprocess_function, batched=True)
tokenized_val = val_set.map(preprocess_function, batched=True)

print("Pre-processing complete for both Train and Validation sets!")


Map:   0%|          | 0/25839 [00:00<?, ? examples/s]

Map:   0%|          | 0/2872 [00:00<?, ? examples/s]

Pre-processing complete for both Train and Validation sets!


In [24]:
import evaluate
import numpy as np

# Load the ROUGE metric
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Decode the predicted tokens back into words
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Decode the labels (ground truth) back into words
    # -100 is a special ID used to tell the model "ignore this" during loss calculation
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Calculate ROUGE scores
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    return {k: round(v, 4) for k, v in result.items()}

In [25]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, T5ForConditionalGeneration

# 1. Load the actual Model (The "Brain")
model = T5ForConditionalGeneration.from_pretrained(model_checkpoint).to(device)

# 2. Data Collator: A helper that handles dynamic padding for the GPU
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 3. Define the Training Rules
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_news_results",
    eval_strategy="epoch",            # Changed from evaluation_strategy
    learning_rate=2e-5,               # The "step size" for optimization
    per_device_train_batch_size=8,    # How many articles to process at once on the GPU
    per_device_eval_batch_size=8,
    weight_decay=0.01,                # Prevents the model from just "memorizing" data (Overfitting)
    save_total_limit=3,               # Only keep the 3 best versions of the model
    num_train_epochs=3,               # Look at the whole dataset 3 times
    predict_with_generate=True,       # Required for summarization tasks
    fp16=True,                        # Use "Mixed Precision" (uses less VRAM, runs 2x faster on T4)
    push_to_hub=False,
    report_to="none"
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [26]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # Your new 90% slice
    eval_dataset=tokenized_val,    # Your new 10% slice (Never seen by the model)
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
# This is it. The big red button.
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,1.943231,1.733040,0.254800,0.107400,0.206700,0.206700
2,1.930013,1.720572,0.253300,0.106700,0.206000,0.206100
3,1.908334,1.715590,0.252900,0.105800,0.205200,0.205200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9690, training_loss=1.9516488593921326, metrics={'train_runtime': 3117.382, 'train_samples_per_second': 24.866, 'train_steps_per_second': 3.108, 'total_flos': 1.0491290424705024e+16, 'train_loss': 1.9516488593921326, 'epoch': 3.0})

In [27]:
# Create a directory for the final model
model_path = "./final_news_t5_model"

# Save the model and the tokenizer
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model saved to {model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./final_news_t5_model


In [17]:
def generate_headline(text):
    # 1. Prepare the input with the prefix
    input_text = "summarize: " + text
    
    # 2. Tokenize and move to GPU
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    # 3. Generate the output tokens
    outputs = model.generate(
    inputs["input_ids"], 
    max_length=20,           # Headlines are usually short
    min_length=5,            # Don't let it be just one word
    num_beams=4, 
    no_repeat_ngram_size=2,  # Stops repetitive phrases
    length_penalty=0.8,      # Gently pushes the model to be more concise
    early_stopping=True
)
    
    # 4. Decode back to English
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# TEST IT: Copy-paste a small paragraph from a recent news article below
test_article = """ 
The space agency NASA has successfully landed a new rover on Mars to search for signs of ancient life. 
The mission, which took seven months to travel through space, is the most ambitious attempt 
to explore the Red Planet to date.
"""

print(f"Generated Headline: {generate_headline(test_article)}")

Generated Headline: NASA successfully landed a new rover on Mars to search for signs of ancient life


In [ ]:
import shutil
from google.colab import files

# Zip the folder
shutil.make_archive("t5_headline_model", 'zip', "./final_news_t5_model")

# Download to your local machine
files.download("t5_headline_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>